# Інтерактивне спряження двох прямих

Це геометрично правильна модель спряження. Дуга **безпосередньо з'єднує дві прямі** у точках дотику P₁ та P₂.

При зміні **R** або **кута** перебудовується вся геометрична конструкція.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider


# ============================================================
# АЛГОРИТМ ПОБУДОВИ СПРЯЖЕННЯ ДВОХ ПРЯМИХ
# ============================================================

def draw_fillet(R=2.0, ANGLE=90):

    alpha = np.radians(ANGLE)

    # --------------------------------------------------------
    # 1. Вершина кута та напрямки двох прямих
    # --------------------------------------------------------

    B = np.array([0.0, 0.0])

    # Обидві прямі виходять із B
    d1 = np.array([1.0, 0.0])
    d2 = np.array([np.cos(alpha), np.sin(alpha)])

    # --------------------------------------------------------
    # 2. Бісектриса кута
    # --------------------------------------------------------

    bisector = d1 + d2
    bisector = bisector / np.linalg.norm(bisector)

    # --------------------------------------------------------
    # 3. Геометрія спряження
    #
    # Відстань від вершини до точки дотику:
    #     t = R / tan(alpha / 2)
    #
    # Відстань від вершини до центра:
    #     BO = R / sin(alpha / 2)
    # --------------------------------------------------------

    t = R / np.tan(alpha / 2)
    BO = R / np.sin(alpha / 2)

    # Точки дотику
    P1 = B + d1 * t
    P2 = B + d2 * t

    # Центр дуги
    O = B + bisector * BO

    # --------------------------------------------------------
    # 4. Відстань від точки до прямої
    # --------------------------------------------------------

    def point_to_line_distance(P, A, direction):
        v = P - A
        return abs(direction[0] * v[1] - direction[1] * v[0])

    d_to_1 = point_to_line_distance(O, B, d1)
    d_to_2 = point_to_line_distance(O, B, d2)

    # --------------------------------------------------------
    # 5. Довжина відображуваних прямих
    # --------------------------------------------------------

    L = max(t + 2.0, 5.0)

    A = B + d1 * L
    C = B + d2 * L

    # --------------------------------------------------------
    # 6. Побудова дуги
    # --------------------------------------------------------

    a1 = np.arctan2(P1[1] - O[1], P1[0] - O[0])
    a2 = np.arctan2(P2[1] - O[1], P2[0] - O[0])

    # Коротка дуга між P1 та P2
    delta = (a2 - a1) % (2 * np.pi)

    if delta > np.pi:
        a1, a2 = a2, a1 + 2 * np.pi
    else:
        a2 = a1 + delta

    angles = np.linspace(a1, a2, 200)

    x_arc = O[0] + R * np.cos(angles)
    y_arc = O[1] + R * np.sin(angles)

    # ========================================================
    # 7. ВІЗУАЛІЗАЦІЯ
    # ========================================================

    fig, ax = plt.subplots(figsize=(10, 7))

    # --------------------------------------------------------
    # Початкові прямі — пунктиром
    # --------------------------------------------------------

    ax.plot(
        [A[0], B[0]],
        [A[1], B[1]],
        '--',
        linewidth=1.2,
        label='Початкова пряма 1'
    )

    ax.plot(
        [B[0], C[0]],
        [B[1], C[1]],
        '--',
        linewidth=1.2,
        label='Початкова пряма 2'
    )

    # --------------------------------------------------------
    # Реальні частини прямих після "Trim"
    # --------------------------------------------------------

    ax.plot(
        [A[0], P1[0]],
        [A[1], P1[1]],
        linewidth=3
    )

    ax.plot(
        [P2[0], C[0]],
        [P2[1], C[1]],
        linewidth=3
    )

    # --------------------------------------------------------
    # ДУГА СПРЯЖЕННЯ
    # Її кінці точно збігаються з P1 та P2
    # --------------------------------------------------------

    ax.plot(
        x_arc,
        y_arc,
        linewidth=5,
        label=f'Спряження R = {R:.1f}'
    )

    # --------------------------------------------------------
    # Радіуси OP1 та OP2
    # --------------------------------------------------------

    ax.plot(
        [O[0], P1[0]],
        [O[1], P1[1]],
        ':',
        linewidth=1.5
    )

    ax.plot(
        [O[0], P2[0]],
        [O[1], P2[1]],
        ':',
        linewidth=1.5
    )

    # --------------------------------------------------------
    # Бісектриса
    # --------------------------------------------------------

    E = B + bisector * BO * 1.25

    ax.plot(
        [B[0], E[0]],
        [B[1], E[1]],
        '-.',
        linewidth=1.2,
        label='Бісектриса'
    )

    # --------------------------------------------------------
    # Точки
    # --------------------------------------------------------

    ax.scatter(
        [P1[0], P2[0]],
        [P1[1], P2[1]],
        s=65,
        zorder=5,
        label='Точки дотику'
    )

    ax.scatter(
        O[0],
        O[1],
        s=65,
        zorder=5,
        label='Центр O'
    )

    # Підписи
    ax.text(B[0] + 0.12, B[1] - 0.35, 'B — вершина', fontsize=11)
    ax.text(P1[0] - 0.1, P1[1] - 0.35, 'P₁', fontsize=11)
    ax.text(P2[0] + 0.12, P2[1] + 0.08, 'P₂', fontsize=11)
    ax.text(O[0] + 0.12, O[1] + 0.12, 'O', fontsize=11)

    # --------------------------------------------------------
    # Контроль геометричної умови
    # --------------------------------------------------------

    ax.text(
        0.02,
        0.98,
        f'R = {R:.2f}\n'
        f'кут = {ANGLE}°\n'
        f'BO = {BO:.2f}\n'
        f'd(O, пряма 1) = {d_to_1:.2f}\n'
        f'd(O, пряма 2) = {d_to_2:.2f}',
        transform=ax.transAxes,
        va='top',
        fontsize=10,
        bbox=dict(boxstyle='round,pad=0.5', alpha=0.9)
    )

    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.25)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

    ax.set_title(
        'Алгоритмічна побудова спряження двох прямих'
    )

    ax.legend(loc='best')

    plt.show()


# ============================================================
# КЕРУВАННЯ
# СТУДЕНТ ЗМІНЮЄ ТІЛЬКИ R ТА КУТ
# ============================================================

interact(
    draw_fillet,

    R=FloatSlider(
        value=2.0,
        min=0.5,
        max=5.0,
        step=0.1,
        description='Радіус R:',
        continuous_update=True
    ),

    ANGLE=IntSlider(
        value=90,
        min=20,
        max=160,
        step=5,
        description='Кут:',
        continuous_update=True
    )
);


## Що перевіряє алгоритм

Для правильного спряження центр **O** має бути на бісектрисі кута, а відстані від **O** до обох прямих мають дорівнювати заданому радіусу **R**.

Тому при зміні повзунків змінюються одночасно центр, точки дотику та дуга. Це вже не окремо намальована крива, а **результат геометричного алгоритму**.